# Diagnostica — Fonti, Cataloghi, Dataset

Vista unificata dello stato del Source Observatory:
- **Fonti**: radar health, quante monitorate vs inventariate
- **Cataloghi**: item count, delta, segnali di drift
- **Dataset**: copertura source_check, intake score, reachability, granularità
- **Copertura**: % inventario processato, gap

**Data**: 2026-06-14

In [1]:
import json

from lab_connectors.duckdb import gcs_connect

# ── Path ──
RADAR_PATH = "../data/radar/radar_summary.json"
SIGNALS_PATH = "../data/catalog/catalog_signals.json"
S3_INV = "s3://dataciviclab-clean/catalog_inventory/catalog_inventory_latest.parquet"
S3_SCR = "s3://dataciviclab-clean/catalog_inventory/source-check/source_check_results.parquet"

# ── Carica dati locali (git) ──
with open(RADAR_PATH) as f:
    radar = json.load(f)
with open(SIGNALS_PATH) as f:
    signals = json.load(f)

# ── Carica dati da S3 ──
with gcs_connect(S3_INV) as con:
    df_inv = con.execute("SELECT * FROM read_parquet(?)", [S3_INV]).fetchdf()
with gcs_connect(S3_SCR) as con:
    df_scr = con.execute("SELECT * FROM read_parquet(?)", [S3_SCR]).fetchdf()

print(f"Radar: {radar['sources_total']} fonti")
print(f"Inventory: {len(df_inv)} rows, {df_inv['source_id'].nunique()} fonti")
print(f"Source-check: {len(df_scr)} rows, {df_scr['source_id'].nunique()} fonti")
print(f"Signals: {signals['sources_checked']} fonti controllate")

Radar: 32 fonti
Inventory: 15075 rows, 28 fonti
Source-check: 10157 rows, 26 fonti
Signals: 28 fonti controllate


---
## 1. Fonti — Radar Health

In [2]:
print("═" * 60)
print("  RADAR HEALTH")
print("═" * 60)
print(f"\nGREEN:  {radar['status_counts'].get('GREEN', 0)}")
print(f"YELLOW: {radar['status_counts'].get('YELLOW', 0)}")
print(f"RED:    {radar['status_counts'].get('RED', 0)}")
print(f"Persistent RED: {radar['persistent_red']}\n")

print("Fonti non VERDI:")
for s in radar["sources"]:
    if s["status"] != "GREEN":
        print(f"  🔴 {s['id']:<25s} {s['status']:<8s} {s.get('note', '')[:80]}")

# Fonti radar MA non in inventory
radar_ids = {s["id"] for s in radar["sources"]}
inv_ids = set(df_inv["source_id"].unique()) if not df_inv.empty else set()
not_inventoried = radar_ids - inv_ids
if not_inventoried:
    print(f"\nFonti non inventariate ({len(not_inventoried)}):")
    for sid in sorted(not_inventoried):
        s = next((s for s in radar["sources"] if s["id"] == sid), {})
        print(f"  {sid:<25s} protocol={s.get('protocol', '?'):<10s}")

inventoried_not_radar = inv_ids - radar_ids
if inventoried_not_radar:
    print(f"\nIn inventory ma non in radar ({len(inventoried_not_radar)}): {inventoried_not_radar}")

════════════════════════════════════════════════════════════
  RADAR HEALTH
════════════════════════════════════════════════════════════

GREEN:  31
YELLOW: 1
RED:    0
Persistent RED: 0

Fonti non VERDI:
  🔴 istat_sdmx                YELLOW   Timeout (ReadTimeout)

Fonti non inventariate (4):
  dait                      protocol=html      
  inail_opendata            protocol=aem       
  noipa_sparql              protocol=sparql    
  terna_opendata            protocol=rest      


---
## 2. Cataloghi — Item count e segnali

In [3]:
print("═" * 60)
print("  CATALOGHI — Item per fonte")
print("═" * 60)

src_counts = df_inv.groupby(["source_id", "protocol"]).size().reset_index(name="items")
src_counts = src_counts.sort_values("items", ascending=False)
print(src_counts.to_string(index=False))

print(f"\nTotale item inventario: {len(df_inv)}")

════════════════════════════════════════════════════════════
  CATALOGHI — Item per fonte
════════════════════════════════════════════════════════════
            source_id protocol  items
           istat_sdmx     sdmx   4874
             openbdap     ckan   3817
                 inps     ckan   2323
          opencivitas     html    703
         opencoesione     ckan    561
               openga     ckan    436
         mim_opendata     html    372
          unioncamere     ckan    371
            mimit_rna     ckan    370
         dati_cultura   sparql    213
            mef_irpef     html    147
          dati_camera   sparql    104
          dati_senato   sparql     98
      lavoro_opendata     ckan     83
         mit_opendata     ckan     70
                 anac     ckan     70
            mur_ustat     ckan     69
    ispra_linked_data   sparql     69
                 agcm     ckan     53
     ministero_salute     ckan     51
                 aifa     html     42
             

In [4]:
print("═" * 60)
print("  SEGNALI DI DRIFT")
print("═" * 60)

for sig in signals["signals"]:
    if sig["signal_type"] != "no signal":
        print(f"\n{sig['source']:<22s} {sig['signal_type']:<20s}")
        print(f"    {sig.get('detail', '')[:150]}")
        print(f"    azione: {sig['suggested_action']}")

════════════════════════════════════════════════════════════
  SEGNALI DI DRIFT
════════════════════════════════════════════════════════════

mim_opendata           csv_magnet          
    1116 link data (CSV 372, JSON 372, XML 372), years 2015-2026 — top prefixes: INFANZIA=192, ALUCORSO=120, SCUANAGR=66, SCUANAAU=66, ALUITAST=60
    azione: catalog-watch-ready

mef_irpef              csv_magnet          
    147 link data (CSV 81, ZIP 66)
    azione: catalog-watch-ready

opencivitas            csv_magnet          
    748 link data (ZIP 748), years 2010-2025 — top prefixes: 2010=90, 2013=72, Metadati=64, 2022=63, 2018=59
    azione: catalog-watch-ready

aifa                   csv_magnet          
    42 link data (CSV 38, XML 1, ZIP 3), years 2010-2027 — top prefixes: provvedimenti=8, Classe=4, sc=3, elenco=2, Elenco=2
    azione: catalog-watch-ready

giustizia_statistiche  csv_magnet          
    9 link data (XLS 9), years 2014-2025 — top prefixes: Indicatori=2, Durata=2, Civile=1,

---
## 3. Dataset — Source-check coverage

---
## 2b. PAQA — Public Administration Quality Assessment

Quality scores strutturali e semantici per ogni CSV profilato.
Disponibile da Giugno 2026 (toolkit v1.36.0+).


In [5]:
print("═" * 60)
print("  PANORAMICA PAQA")
print("═" * 60)

total = len(df_scr)
with_paqa = df_scr["paqa_score"].notna().sum()
without_paqa = total - with_paqa
print(f"Item con PAQA score:  {with_paqa:>5}  ({with_paqa / total * 100:.1f}%)")
print(f"Item senza PAQA:      {without_paqa:>5}  ({without_paqa / total * 100:.1f}%)")

if with_paqa > 0:
    print(f"Score medio:          {df_scr['paqa_score'].mean():>5.1f}")
    print(f"Score mediano:        {df_scr['paqa_score'].median():>5.0f}")
    print(f"Dev std:              {df_scr['paqa_score'].std():>5.1f}")
    print(f"Min:                  {df_scr['paqa_score'].min():>5.0f}")
    print(f"Max:                  {df_scr['paqa_score'].max():>5.0f}")

    print("\n  VERDETTI")
    verdicts = df_scr["paqa_verdict"].value_counts()
    for v, c in verdicts.items():
        print(f"    {str(v):<15s} {c:>5}  ({c / with_paqa * 100:.1f}%)")

    print(f"\n  ITEM CON FLAG:    {df_scr['paqa_flags'].notna().sum():>5}")
    print(f"  ITEM CON ONTOLOGIE: {df_scr['paqa_ontologies'].notna().sum():>5}")
    print(f"  SAMPLED:            {df_scr['paqa_sampled'].sum():>5}")

════════════════════════════════════════════════════════════
  PANORAMICA PAQA
════════════════════════════════════════════════════════════
Item con PAQA score:   1112  (10.9%)
Item senza PAQA:       9045  (89.1%)
Score medio:           92.8
Score mediano:           92
Dev std:                3.2
Min:                     78
Max:                     98

  VERDETTI
    buona             974  (87.6%)
    scarsa             90  (8.1%)
    accettabile        48  (4.3%)

  ITEM CON FLAG:      889
  ITEM CON ONTOLOGIE:   912
  SAMPLED:              248


In [6]:
import pandas as pd

print("═" * 60)
print("  PAQA PER FONTE")
print("═" * 60)

src_paqa = (
    df_scr[df_scr["paqa_score"].notna()]
    .groupby("source_id")
    .agg(
        items=("paqa_score", "count"),
        avg_score=("paqa_score", "mean"),
        med_score=("paqa_score", "median"),
        min_score=("paqa_score", "min"),
        max_score=("paqa_score", "max"),
        std_score=("paqa_score", "std"),
    )
    .reset_index()
)
src_paqa = src_paqa.sort_values("avg_score", ascending=False)

print(f"{'Fonte':22s} {'Item':>5s} {'Avg':>5s} {'Med':>4s} {'Min':>4s} {'Max':>4s} {'Std':>5s}")
print("-" * 52)
for _, r in src_paqa.iterrows():
    std_str = f"{r['std_score']:.1f}" if pd.notna(r["std_score"]) else "N/A"
    print(
        f"{r['source_id']:22s} {r['items']:5.0f} {r['avg_score']:5.0f} {r['med_score']:4.0f} {r['min_score']:4.0f} {r['max_score']:4.0f} {std_str:>5s}"
    )

════════════════════════════════════════════════════════════
  PAQA PER FONTE
════════════════════════════════════════════════════════════
Fonte                   Item   Avg  Med  Min  Max   Std
----------------------------------------------------
inps                      43    96   95   90   98   2.1
openga                   159    96   95   90   98   1.5
pagopa                     8    95   95   92   98   2.5
ministero_salute           2    95   95   95   95   0.0
ministero_interno         38    94   95   88   98   3.4
aci                        4    94   94   94   94   0.0
unioncamere              279    94   95   78   98   3.4
mit_opendata              25    92   92   83   98   4.4
mef_irpef                 79    92   92   88   96   1.7
mim_opendata             392    91   92   81   95   2.9
lavoro_opendata           83    90   90   90   90   0.0


In [7]:
import ast

print("═" * 60)
print("  PAQA — FLAG")
print("═" * 60)

flags_series = df_scr[
    df_scr["paqa_flags"].notna() & (df_scr["paqa_flags"] != "[]") & (df_scr["paqa_flags"] != "")
]

# Parse e conta flag (ogni riga può avere multipli flag in formato JSON array string)

flag_counter = {}
for raw in flags_series["paqa_flags"]:
    try:
        flags = ast.literal_eval(raw) if isinstance(raw, str) else raw
    except Exception:
        flags = []
    key = ", ".join(flags) if flags else "empty"
    flag_counter[key] = flag_counter.get(key, 0) + 1

sorted_flags = sorted(flag_counter.items(), key=lambda x: -x[1])
print(f"{'Flag combo':55s} {'Item':>5s} {'%':>6s}")
print("-" * 68)
for combo, cnt in sorted_flags[:15]:
    print(f"  [{combo:50s}] {cnt:5d}  ({cnt / flags_series.shape[0] * 100:5.1f}%)")

if len(sorted_flags) > 15:
    print(f"  ... e altre {len(sorted_flags) - 15} combinazioni")

════════════════════════════════════════════════════════════
  PAQA — FLAG
════════════════════════════════════════════════════════════


Flag combo                                               Item      %
--------------------------------------------------------------------
  [non_iso_dates                                     ]   390  ( 43.9%)
  [column_naming                                     ]   320  ( 36.0%)
  [inconsistent_columns, column_naming               ]    87  (  9.8%)
  [column_naming, non_iso_dates                      ]    44  (  4.9%)
  [duplicate_columns, column_naming                  ]    16  (  1.8%)
  [encoding_issues, non_iso_dates                    ]    15  (  1.7%)
  [high_missing_rate                                 ]     3  (  0.3%)
  [inconsistent_columns, non_iso_dates               ]     3  (  0.3%)
  [encoding_issues, column_naming                    ]     2  (  0.2%)
  [encoding_issues                                   ]     2  (  0.2%)
  [duplicate_columns, high_missing_rate, column_naming]     2  (  0.2%)
  [duplicate_columns, column_naming, non_iso_dates   ]     2  (  0.2%)
  [duplic

In [8]:
print("═" * 60)
print("  PAQA — ONTOLOGIE RILEVATE")
print("═" * 60)

onto_series = df_scr[
    df_scr["paqa_ontologies"].notna()
    & (df_scr["paqa_ontologies"] != "{}")
    & (df_scr["paqa_ontologies"] != "")
]

onto_counter = {}
for raw in onto_series["paqa_ontologies"]:
    try:
        onto = ast.literal_eval(raw) if isinstance(raw, str) else raw
    except Exception:
        onto = {}
    # Extract ontology codes
    codes = sorted(onto.keys()) if onto else ["empty"]
    key = ", ".join(codes)
    onto_counter[key] = onto_counter.get(key, 0) + 1

sorted_onto = sorted(onto_counter.items(), key=lambda x: -x[1])
print(f"{'Ontologia/e':55s} {'Item':>5s}")
print("-" * 62)
for combo, cnt in sorted_onto[:12]:
    print(f"  {combo:55s} {cnt:5d}")

if len(sorted_onto) > 12:
    print(f"  ... e altre {len(sorted_onto) - 12} combinazioni")

print(f"\nItem con ontologia/e: {onto_series.shape[0]}")

════════════════════════════════════════════════════════════
  PAQA — ONTOLOGIE RILEVATE
════════════════════════════════════════════════════════════


Ontologia/e                                              Item
--------------------------------------------------------------
  TI                                                        195
  CPV, TI                                                   135
  ISTAT, TI                                                 120
  CLV, ISTAT, TI                                             80
  CLV, COV, ISTAT, TI                                        59
  CLV                                                        58
  CLV, ISTAT                                                 46
  ISTAT                                                      44
  CLV, ISTAT, PC                                             27
  QB, TI                                                     23
  CPV, QB, TI                                                22
  QB                                                         20
  ... e altre 27 combinazioni

Item con ontologia/e: 912


In [9]:
import pandas as pd

print("═" * 60)
print("  PAQA vs INTAKE SCORE")
print("═" * 60)

both = df_scr[df_scr["paqa_score"].notna() & df_scr["intake_score"].notna()].copy()
if len(both) > 0:
    corr = both["paqa_score"].corr(both["intake_score"])
    print(f"Correlazione PAQA ↔ Intake: {corr:.3f}")
    print(f"(su {len(both)} item con entrambi gli score)")

    incrocio = (
        both.groupby(
            pd.cut(both["paqa_score"], bins=[0, 70, 90, 100], labels=["<70", "70-90", "90-100"])
        )
        .agg(n=("intake_score", "count"), avg_intake=("intake_score", "mean"))
        .reset_index()
    )
    print(f"\n  {'Fascia PAQA':12s} {'N':>5s} {'Avg Intake':>10s}")
    print("  " + "-" * 30)
    for _, r in incrocio.iterrows():
        print(f"  {str(r['paqa_score']):12s} {r['n']:5.0f} {r['avg_intake']:10.1f}")

    # Top item: alto PAQA + alto intake
    top = both.nlargest(5, "intake_score")[["source_id", "title", "intake_score", "paqa_score"]]
    print("\n  Top 5 per intake_score (con PAQA):")
    for _, r in top.iterrows():
        print(
            f"    {r['source_id']:<15s} intake={r['intake_score']:.0f} paqa={r['paqa_score']:.0f}  {str(r.get('title', ''))[:60]}"
        )
else:
    print("Nessun item con entrambi gli score disponibile")

════════════════════════════════════════════════════════════
  PAQA vs INTAKE SCORE
════════════════════════════════════════════════════════════
Correlazione PAQA ↔ Intake: 0.268
(su 1112 item con entrambi gli score)



  Fascia PAQA      N Avg Intake
  ------------------------------
  70-90          213       53.5
  90-100         899       65.2

  Top 5 per intake_score (con PAQA):
    mit_opendata    intake=100 paqa=98  Posti Barca
    mit_opendata    intake=100 paqa=90  Storico interventi previsti dai Contratti di Programma RFI e
    mit_opendata    intake=100 paqa=85  Interventi delle Autorità di Sistema Portuale
    ministero_interno intake=100 paqa=92  Popolazione residente
    mit_opendata    intake=100 paqa=83  Interventi previsti dai Contratti di Programma RFI e ANAS


In [10]:
print("═" * 60)
print("  SAMPLED — PREVIEW TRONCA")
print("═" * 60)

sampled_true = df_scr[df_scr["paqa_sampled"]]
sampled_false = df_scr[~df_scr["paqa_sampled"]]
print(f"Sampled (preview tronca, check S6/S12 disabilitati): {len(sampled_true)}")
print(f"Full preview:                                         {len(sampled_false)}")

if len(sampled_true) > 0 and len(sampled_false) > 0:
    score_sampled = sampled_true["paqa_score"].mean()
    score_full = sampled_false["paqa_score"].mean()
    print(f"Score medio sampled:   {score_sampled:.1f}")
    print(f"Score medio full:      {score_full:.1f}")
    print(f"Differenza:            {score_full - score_sampled:+.1f}")

════════════════════════════════════════════════════════════
  SAMPLED — PREVIEW TRONCA
════════════════════════════════════════════════════════════


Sampled (preview tronca, check S6/S12 disabilitati): 248
Full preview:                                         864
Score medio sampled:   90.8
Score medio full:      93.4
Differenza:            +2.6


In [11]:
print("═" * 60)
print("  COPERTURA SOURCE-CHECK")
print("═" * 60)

# Merge inventory + source_check
inv_count = df_inv.groupby("source_id").size().reset_index(name="inventory_items")
scr_count = df_scr.groupby("source_id").size().reset_index(name="scored_items")
coverage = inv_count.merge(scr_count, on="source_id", how="outer").fillna(0)
coverage["pct"] = (
    coverage["scored_items"] / coverage["inventory_items"].clip(lower=1) * 100
).round(1)
coverage = coverage.astype({"inventory_items": int, "scored_items": int})
coverage = coverage.sort_values("inventory_items", ascending=False)

print(f"{'Fonte':<22s} {'Inventory':>10s} {'Scored':>8s} {'Coperto':>8s}")
print("-" * 50)
for _, r in coverage.iterrows():
    print(
        f"{r['source_id']:<22s} {r['inventory_items']:>10d} {r['scored_items']:>8.0f} {r['pct']:>7.1f}%"
    )
print(f"\nTotale inventory: {coverage['inventory_items'].sum():.0f}")
print(f"Totale scored:    {coverage['scored_items'].sum():.0f}")
print(
    f"Copertura media:  {coverage['scored_items'].sum() / coverage['inventory_items'].sum() * 100:.1f}%"
)

════════════════════════════════════════════════════════════
  COPERTURA SOURCE-CHECK
════════════════════════════════════════════════════════════
Fonte                   Inventory   Scored  Coperto
--------------------------------------------------
istat_sdmx                   4874     3833    78.6%
openbdap                     3817     1927    50.5%
inps                         2323     1427    61.4%
opencivitas                   703      384    54.6%
opencoesione                  561      518    92.3%
openga                        436      364    83.5%
mim_opendata                  372      427   114.8%
unioncamere                   371      371   100.0%
mimit_rna                     370       85    23.0%
dati_cultura                  213      213   100.0%
mef_irpef                     147      119    81.0%
dati_camera                   104      104   100.0%
dati_senato                    98        0     0.0%
lavoro_opendata                83       83   100.0%
mit_opendata          

In [12]:
print("═" * 60)
print("  QUALITA' DATASET")
print("═" * 60)

total = len(df_scr)
print(f"Item totali:            {total:>6}")
print(
    f"Candidati intake:       {df_scr['intake_candidate'].sum():>6} ({df_scr['intake_candidate'].sum() / total * 100:.0f}%)"
)
print(
    f"Needs review:           {df_scr['needs_review'].sum():>6} ({df_scr['needs_review'].sum() / total * 100:.0f}%)"
)
print(
    f"Reachable:              {df_scr['reachable'].sum():>6} ({df_scr['reachable'].sum() / total * 100:.0f}%)"
)
print(
    f"Con colonne profilate:  {df_scr['columns'].notna().sum():>6} ({df_scr['columns'].notna().sum() / total * 100:.0f}%)"
)
print(
    f"Con join_keys:          {df_scr['join_keys'].notna().sum():>6} ({df_scr['join_keys'].notna().sum() / total * 100:.0f}%)"
)
print(f"Intake score medio:     {df_scr['intake_score'].mean():6.1f}")
if "joinability_score" in df_scr.columns:
    print(f"Joinability score medio: {df_scr['joinability_score'].mean():6.1f}")

════════════════════════════════════════════════════════════
  QUALITA' DATASET
════════════════════════════════════════════════════════════
Item totali:             10157
Candidati intake:         1437 (14%)
Needs review:             8247 (81%)
Reachable:                2863 (28%)
Con colonne profilate:    2142 (21%)
Con join_keys:            1275 (13%)
Intake score medio:       26.5
Joinability score medio:    5.2


In [13]:
print("═" * 60)
print("  GRANULARITA'")
print("═" * 60)
gran = df_scr["granularity"].value_counts()
for g, c in gran.items():
    print(f"  {g:<20s} {c:>5} ({c / total * 100:.0f}%)")

print("\n  FORMATI")
fmt = df_scr["resource_format"].value_counts()
for f, c in fmt.head(8).items():
    print(f"  {str(f):<10s} {c:>5} ({c / total * 100:.0f}%)")

════════════════════════════════════════════════════════════
  GRANULARITA'
════════════════════════════════════════════════════════════
  non_determinato       7030 (69%)
  regione               1664 (16%)
  provincia              492 (5%)
  comune                 392 (4%)
  nazionale              352 (3%)
  europeo                108 (1%)
  mondiale                 2 (0%)

  FORMATI
  CSV         5162 (51%)
  SDMX        3268 (32%)
  XLS          699 (7%)
  ZIP          437 (4%)
               317 (3%)
  XML          130 (1%)
  JSON          12 (0%)
  XLSX           8 (0%)


---
## 4. Sintesi

Cose da fare:
- Fonti non inventariate → valutare se aggiungere a catalog-watch
- Fonti con segnali di drift → verificare
- Source_check da completare su fonti a bassa copertura
- Dataset con intake_score alto ma senza join_keys → potenziali falsi negativi
- Fonti con PAQA basso (< 85) → verificare quality check


In [14]:
print("═" * 60)
print("  COSE DA FARE")
print("═" * 60)

# 1. Fonti non inventariate
if not_inventoried:
    print(f"\n📋 Fonti da inventariare: {', '.join(sorted(not_inventoried))}")

# 2. Fonti con segnali
signals_active = [s for s in signals["signals"] if s["signal_type"] != "no signal"]
if signals_active:
    print(f"\n📋 Fonti con segnali attivi ({len(signals_active)}):")
    for s in signals_active:
        print(
            f"     {s['source']:<22s} {s['signal_type']:<20s} {s.get('suggested_action', '')[:60]}"
        )

# 3. Fonti con bassa copertura source-check
low_cov = coverage[coverage["pct"] < 50].sort_values("pct")
if not low_cov.empty:
    print(f"\n📋 Fonti con copertura < 50% ({len(low_cov)}):")
    for _, r in low_cov.iterrows():
        print(
            f"     {r['source_id']:<22s} {r['pct']:.0f}% ({r['scored_items']:.0f}/{r['inventory_items']:.0f})"
        )

# 4. Intake alto senza join_keys
high_intake_no_keys = df_scr[(df_scr["intake_score"] >= 50) & (df_scr["join_keys"].isna())].copy()
if len(high_intake_no_keys) > 0:
    print(f"\n📋 Dataset con intake >= 50 MA senza join_keys ({len(high_intake_no_keys)}):")
    top = high_intake_no_keys.sort_values("intake_score", ascending=False).head(5)
    for _, r in top.iterrows():
        print(
            f"     {r['source_id']:<15s} score={r['intake_score']:.0f} {str(r.get('title', ''))[:50]}"
        )

════════════════════════════════════════════════════════════
  COSE DA FARE
════════════════════════════════════════════════════════════

📋 Fonti da inventariare: dait, inail_opendata, noipa_sparql, terna_opendata

📋 Fonti con segnali attivi (6):
     mim_opendata           csv_magnet           catalog-watch-ready
     mef_irpef              csv_magnet           catalog-watch-ready
     opencivitas            csv_magnet           catalog-watch-ready
     aifa                   csv_magnet           catalog-watch-ready
     giustizia_statistiche  csv_magnet           low signal
     cortecostituzionale    csv_magnet           catalog-watch-ready

📋 Fonti con copertura < 50% (10):
     dati_senato            0% (0/98)
     ispra_linked_data      0% (0/69)
     agcm                   8% (4/53)
     aci                    11% (4/35)
     anac                   19% (13/70)
     agid                   20% (8/40)
     mimit_rna              23% (85/370)
     cortecostituzionale    39% (13/33)


---
## 5. Note

- I dati radar e signals vengono dal repository git (ultimo commit).
- I dati inventory e source_check vengono da S3 (ultimo run CI).
- La diagnostica non modifica nulla — è una vista sola.